In [0]:
import sys
print(sys.version)
print(spark.version)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

#Config 
random.seed(42)

PRACTICE_AREAS  = ["Data & Analytics", "Cloud", "Cybersecurity", "ERP", "Digital"]
SENIORITY_LEVELS = ["Analyst", "Consultant", "Senior Consultant", "Manager", "Senior Manager"]
SKILLS          = ["Python", "SQL", "Databricks", "Azure", "Power BI", "SAP",
                   "Tableau", "Spark", "DevOps", "Machine Learning"]
PROJECT_TYPES   = ["Implementation", "Advisory", "Assessment", "Managed Service"]
NUM_EMPLOYEES   = 100
NUM_PROJECTS    = 30

def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

start_range = datetime(2022, 1, 1)
end_range   = datetime(2024, 12, 31)
today       = datetime(2025, 1, 1)

In [0]:
# Employees
employees = []

for i in range(1, NUM_EMPLOYEES + 1):
    hire_date = random_date(start_range, end_range)
    is_active = random.random() > 0.1
    employees.append({
        "employee_id":   f"EMP{i:03d}",
        "name":          f"Employee_{i}",
        "seniority":     random.choice(SENIORITY_LEVELS),
        "practice_area": random.choice(PRACTICE_AREAS),
        "hire_date":     hire_date.date(),
        "status":        "Active" if is_active else "Inactive",
        "email":         f"employee_{i}@consultancy.com"
    })

employees_df = spark.createDataFrame(employees)
employees_df.write.format("delta").mode("overwrite").saveAsTable("bronze.employees")
print(f"Employees written: {employees_df.count()} rows")
employees_df.show(5)

In [0]:
# Check current catalog and available schemas
print("Current catalog:", spark.catalog.currentCatalog())
print("Current schema:", spark.catalog.currentDatabase())

# List available catalogs
display(spark.sql("SHOW CATALOGS"))

In [0]:

spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

print("Current catalog:", spark.catalog.currentCatalog())
print("Schemas created successfully")
display(spark.sql("SHOW SCHEMAS"))

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

# ── Config
random.seed(42)

spark.sql("USE CATALOG workspace")

PRACTICE_AREAS   = ["MDP", "AIBP", "Advanced Analytics", "Data Governance", "Business Intelligence"]
SENIORITY_LEVELS = ["Analyst", "Consultant", "Senior Consultant", "Manager", "Senior Manager"]
SKILLS           = ["Python", "SQL", "Databricks", "Azure", "Power BI", "SAP",
                    "Tableau", "Spark", "DevOps", "Machine Learning"]
PROJECT_TYPES    = ["Implementation", "Advisory", "Assessment", "Managed Service"]
NUM_EMPLOYEES    = 100
NUM_PROJECTS     = 30

def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

start_range = datetime(2022, 1, 1)
end_range   = datetime(2024, 12, 31)
today       = datetime(2025, 1, 1)

print("Config ready")
print("Catalog:", spark.catalog.currentCatalog())

In [0]:

employees = []

for i in range(1, NUM_EMPLOYEES + 1):
    hire_date = random_date(start_range, end_range)
    is_active = random.random() > 0.1
    employees.append({
        "employee_id":   f"EMP{i:03d}",
        "name":          f"Employee_{i}",
        "seniority":     random.choice(SENIORITY_LEVELS),
        "practice_area": random.choice(PRACTICE_AREAS),
        "hire_date":     hire_date.date(),
        "status":        "Active" if is_active else "Inactive",
        "email":         f"employee_{i}@kubrickgroup.com"
    })

employees_df = spark.createDataFrame(employees)
employees_df.write.format("delta").mode("overwrite").saveAsTable("bronze.employees")
print(f"Employees written: {employees_df.count()} rows")
employees_df.show(5)

In [0]:
# ── Projects ────────────────────────────────────────────────────────────────
projects = []

for i in range(1, NUM_PROJECTS + 1):
    start_date = random_date(start_range, end_range)
    duration   = random.randint(30, 365)
    end_date   = start_date + timedelta(days=duration)
    projects.append({
        "project_id":    f"PRJ{i:03d}",
        "client":        f"Client_{i}",
        "practice_area": random.choice(PRACTICE_AREAS),
        "project_type":  random.choice(PROJECT_TYPES),
        "start_date":    start_date.date(),
        "end_date":      end_date.date(),
        "headcount":     random.randint(2, 10),
        "status":        "Active" if end_date > today else "Completed"
    })

projects_df = spark.createDataFrame(projects)
projects_df.write.format("delta").mode("overwrite").saveAsTable("bronze.projects")
print(f"Projects written: {projects_df.count()} rows")
projects_df.show(5)

In [0]:
# ── Assignments ──────────────────────────────────────────────────────────────
assignments = []
assignment_id = 1

for i in range(1, NUM_PROJECTS + 1):
    project_id  = f"PRJ{i:03d}"
    project     = next(p for p in projects if p["project_id"] == project_id)
    num_assigns = random.randint(2, min(project["headcount"], 8))
    assigned    = random.sample([e["employee_id"] for e in employees], num_assigns)

    for emp_id in assigned:
        start = random_date(project["start_date"], project["end_date"] - timedelta(days=30))
        end   = random_date(start + timedelta(days=14), project["end_date"])
        assignments.append({
            "assignment_id": f"ASN{assignment_id:04d}",
            "employee_id":   emp_id,
            "project_id":    project_id,
            "start_date":    start,
            "end_date":      end,
            "role":          random.choice(SENIORITY_LEVELS),
            "practice_area": project["practice_area"]
        })
        assignment_id += 1

assignments_df = spark.createDataFrame(assignments)
assignments_df.write.format("delta").mode("overwrite").saveAsTable("bronze.assignments")
print(f"Assignments written: {assignments_df.count()} rows")
assignments_df.show(5)

In [0]:
# ── Timesheets ───────────────────────────────────────────────────────────────
timesheets = []
timesheet_id = 1

for assignment in assignments:
    start  = assignment["start_date"]
    end    = assignment["end_date"]
    current = start

    while current < end:
        week_end = current + timedelta(days=7)
        timesheets.append({
            "timesheet_id":  f"TS{timesheet_id:05d}",
            "employee_id":   assignment["employee_id"],
            "project_id":    assignment["project_id"],
            "assignment_id": assignment["assignment_id"],
            "week_start":    current.date() if hasattr(current, 'date') else current,
            "week_end":      week_end.date() if hasattr(week_end, 'date') else week_end,
            "hours_logged":  round(random.uniform(20, 45), 1),
            "billable":      random.random() > 0.1
        })
        timesheet_id += 1
        current = week_end

timesheets_df = spark.createDataFrame(timesheets)
timesheets_df.write.format("delta").mode("overwrite").saveAsTable("bronze.timesheets")
print(f"Timesheets written: {timesheets_df.count()} rows")
timesheets_df.show(5)

In [0]:
skills = []

for employee in employees:
    num_skills = random.randint(2, 5)
    emp_skills = random.sample(SKILLS, num_skills)
    for skill in emp_skills:
        skills.append({
            "employee_id":   employee["employee_id"],
            "skill":         skill,
            "proficiency":   random.choice(["Beginner", "Intermediate", "Advanced", "Expert"]),
            "certified":     random.random() > 0.6,
            "last_assessed": random_date(start_range, end_range).date()
        })

skills_df = spark.createDataFrame(skills)
skills_df.write.format("delta").mode("overwrite").saveAsTable("bronze.skills")
print(f"Skills written: {skills_df.count()} rows")
skills_df.show(5)

In [0]:
# Bench Periods
bench_periods = []
bench_id = 1

for employee in employees:
    emp_id = employee["employee_id"]
    emp_assignments = sorted(
        [a for a in assignments if a["employee_id"] == emp_id],
        key=lambda x: x["start_date"]
    )

    # Bench period before first assignment
    if emp_assignments:
        first_start = emp_assignments[0]["start_date"]
        if isinstance(first_start, datetime):
            first_start = first_start.date()
        hire_date = employee["hire_date"]
        if (first_start - hire_date).days > 14:
            bench_periods.append({
                "bench_id":     f"BEN{bench_id:04d}",
                "employee_id":  emp_id,
                "start_date":   hire_date,
                "end_date":     first_start,
                "days_on_bench": (first_start - hire_date).days,
                "reason":       "Post-hire"
            })
            bench_id += 1

    # Bench periods between assignments
    for i in range(len(emp_assignments) - 1):
        curr_end  = emp_assignments[i]["end_date"]
        next_start = emp_assignments[i + 1]["start_date"]
        if isinstance(curr_end, datetime):
            curr_end = curr_end.date()
        if isinstance(next_start, datetime):
            next_start = next_start.date()
        gap_days = (next_start - curr_end).days
        if gap_days > 14:
            bench_periods.append({
                "bench_id":      f"BEN{bench_id:04d}",
                "employee_id":   emp_id,
                "start_date":    curr_end,
                "end_date":      next_start,
                "days_on_bench": gap_days,
                "reason":        random.choice(["Between projects", "Training", "Internal"])
            })
            bench_id += 1

bench_df = spark.createDataFrame(bench_periods)
bench_df.write.format("delta").mode("overwrite").saveAsTable("bronze.bench_periods")
print(f"Bench periods written: {bench_df.count()} rows")
bench_df.show(5)

In [0]:
# ── Bronze table summary 
tables = ["employees", "projects", "assignments", "timesheets", "skills", "bench_periods"]

print("=== Bronze Layer Summary ===\n")
for table in tables:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM bronze.{table}").collect()[0][0]
    print(f"bronze.{table}: {count} rows")